In [0]:
%sql
CREATE OR REPLACE VIEW sts_prd.`01_ipd_thering`.g_bowler_sales_order_all_ipd_agg AS
WITH material_set AS (
  SELECT DISTINCT Product, Level1Desc
  FROM hub_prd.g_bpc_finance.v_reltioproduct_hier
  WHERE Level1Desc IN (
    'Alaris System','Hazardous Drug Solutions','IV Solutions',
    'Gravity and Syringe Delivery','IV Access','CME',
    'Alaris Plus','IPD OEM','BD Nexus'
  )
  AND Product IS NOT NULL
),
cfnnumbers AS (
  SELECT ProductLots, ProductScm
  FROM hub_prd.g_isc.v_lots_master
),
-- LOTS customer master to fetch GlobalId for ShipTo
lots_shipto_global AS (
  SELECT cm.CustomerNumber AS ShipToNumber, cm.GlobalId
  FROM hub_prd.s_control_tower.customer_master_delta_lots_semantic cm
  WHERE cm.System = 'LOTS'
),
-- LOTS site master to check if ShipToNumber is internal
lots_site_master AS (
  SELECT DISTINCT CustomerNumber AS ShipToNumber
  FROM hub_prd.s_isc.site_master_all_sources
),
sales_orders AS (
  SELECT
    so.*,
    CASE
      WHEN so.SourceSystem = 'LOT' THEN COALESCE(cfn.ProductLots, so.MaterialNumber)  -- LOT first, else raw
      WHEN so.MaterialNumberHarmonized IS NOT NULL AND so.MaterialNumberHarmonized <> '' THEN so.MaterialNumberHarmonized
      ELSE so.MaterialNumber
    END AS MaterialNumberwithcfn
  FROM hub_prd.s_shared.sales_orders_all_sources so
  LEFT JOIN cfnnumbers cfn
    ON so.SourceSystem = 'LOT' AND so.MaterialNumber = cfn.ProductScm
)

SELECT
    -- group-by columns
    so.CreateDate,
    so.SourceSystem,
    so.OrderNumber,
    so.OrderType,
    so.ShipToNumber,
    so.SoldToNumber,
    so.MaterialNumber,
    so.MaterialNumberHarmonized,
    so.MaterialNumberwithcfn AS ProductKeyforFinance,
    so.OrderStatusLineLast,
    so.OrderStatusLineNext,
    so.OrderStatusHeader,
    so.OrderStatusLineHarmonized,
    so.OrderStatusHeaderHarmonized,
    CASE WHEN so.SourceSystem = 'LOT' THEN CASE UPPER(TRIM(so.OrderStatusHeader)) WHEN 'Q' THEN 'NOT RELEVANT' WHEN 'C' THEN 'CANCELLED' WHEN 'P' THEN 'COMPLETELY PROCESSED' WHEN 'A' THEN 'NOT YET PROCESSED' ELSE so.OrderStatusHeaderHarmonized END ELSE so.OrderStatusHeaderHarmonized END AS OrderStatusHeaderHarmonizedValid,
    CASE WHEN so.SourceSystem = 'LOT' THEN CASE UPPER(TRIM(so.OrderStatusLineLast)) WHEN 'Q' THEN 'DRAFT' WHEN 'C' THEN 'CANCELLED' WHEN 'H' THEN 'HOLD' WHEN 'E' THEN 'ERROR' WHEN '' THEN 'COMPLETELY PROCESSED' ELSE so.OrderStatusLineHarmonized END ELSE so.OrderStatusLineHarmonized END AS OrderStatusLineHarmonizedValid,
    so.SalesDistributionChannel,
    so.SalesDistChannelHarmonized,
    CASE 
      WHEN so.SalesDistChannelHarmonized IS NOT NULL AND TRIM(so.SalesDistChannelHarmonized) <> '' THEN so.SalesDistChannelHarmonized 
      WHEN so.SourceSystem = 'LOT' THEN 
        CASE 
          WHEN lg.GlobalId LIKE '9%' 
            OR so.ShipToNumber LIKE 'WH%' 
            OR so.ShipToNumber LIKE 'TSE%' 
            OR so.ShipToNumber LIKE 'PLANT%' 
            OR so.ShipToNumber LIKE 'SCM%' 
            OR so.ShipToNumber LIKE 'REP%' 
            OR so.ShipToNumber LIKE 'REPUKI%' 
            OR so.ShipToNumber LIKE 'DUMMY%' 
            OR so.ShipToNumber LIKE 'PANAMA%' 
            OR so.ShipToNumber LIKE 'BDIRELAND%' 
            OR so.ShipToNumber LIKE 'DEMO%' 
            OR so.ShipToNumber LIKE '*%' 
            OR so.ShipToNumber = "092027010018" 
            OR so.ShipToNumber = "%ONLINE%" 
            OR so.ShipToName LIKE 'BD %' 
            OR so.ShipToName LIKE 'GCS %' 
            OR so.ShipToName LIKE 'BD%' 
            OR so.ShipToName LIKE 'BECTON%' 
            OR sm.ShipToNumber IS NOT NULL 
          THEN 'Internal' 
          ELSE 'External' 
        END 
      ELSE so.SalesDistChannelHarmonized 
    END AS SalesDistChannelHarmonizedValid,
    so.ShipToName,
    so.SoldToName,
    so.Country,
    so.CustomerPo,
    so.OrderUom,
    so.BaseUom,
    so.CurrencyType,
    so.ExchangeRate,
    so.ExchangeDate,
    so.UnitPriceLocal,
    so.UnitPriceUsd,
    so.PromisedDeliveryDate,
    so.RequestedDeliveryDate,
    so.ActualShipDate,
    so.LastChangeDate,
    so.ExtractDate,
    so.LocalCurrency,
    so.ErpCustomerKey,
    so.AgreementNumber,
    so.ContractStartDate,
    so.ContractEndDate,

    -- measures
    SUM(so.OrderQuantityOriginal)       AS OrderQuantityOriginal,
    SUM(so.OrderQuantityBase)           AS OrderQuantityBase,
    SUM(so.CancelledQuantityOriginal)   AS CancelledQuantityOriginal,
    SUM(so.CancelledQuantityBase)       AS CancelledQuantityBase,
    SUM(so.DeliveredQuantityOriginal)   AS DeliveredQuantityOriginal,
    SUM(so.DeliveredQuantityBase)       AS DeliveredQuantityBase,
    SUM(so.OpenQuantityOriginal)        AS OpenQuantityOriginal,
    SUM(so.OpenQuantityBase)            AS OpenQuantityBase,
    SUM(so.BackOrderQuantityOriginal)   AS BackOrderQuantityOriginal,
    SUM(so.BackOrderQuantityBase)       AS BackOrderQuantityBase,
    SUM(so.TotalPriceLocal)             AS TotalPriceLocal,
    SUM(so.TotalPriceUsd)               AS TotalPriceUsd
FROM sales_orders so
INNER JOIN material_set ms
  ON so.MaterialNumberwithcfn = ms.Product
LEFT JOIN lots_shipto_global lg
  ON so.SourceSystem = 'LOT' AND so.ShipToNumber = lg.ShipToNumber
LEFT JOIN lots_site_master sm
  ON so.SourceSystem = 'LOT' AND so.ShipToNumber = sm.ShipToNumber   -- ✅ join to site master for internal classification
WHERE year(so.CreateDate) >= year(current_date()) - 2
GROUP BY
    so.CreateDate,
    so.SourceSystem,
    so.OrderNumber,
    so.OrderType,
    so.ShipToNumber,
    so.SoldToNumber,
    so.MaterialNumber,
    so.MaterialNumberHarmonized,
    so.MaterialNumberwithcfn,
    so.OrderStatusLineLast,
    so.OrderStatusLineNext,
    so.OrderStatusHeader,
    so.OrderStatusLineHarmonized,
    so.OrderStatusHeaderHarmonized,
    OrderStatusHeaderHarmonizedValid,
    OrderStatusLineHarmonizedValid,
    so.SalesDistributionChannel,
    so.SalesDistChannelHarmonized,
    SalesDistChannelHarmonizedValid,
    so.ShipToName,
    so.SoldToName,
    so.Country,
    so.CustomerPo,
    so.OrderUom,
    so.BaseUom,
    so.CurrencyType,
    so.ExchangeRate,
    so.ExchangeDate,
    so.UnitPriceLocal,
    so.UnitPriceUsd,
    so.PromisedDeliveryDate,
    so.RequestedDeliveryDate,
    so.ActualShipDate,
    so.LastChangeDate,
    so.ExtractDate,
    so.LocalCurrency,
    so.ErpCustomerKey,
    so.AgreementNumber,
    so.ContractStartDate,
    so.ContractEndDate;
